In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    784 non-null    object
 1   label   784 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 12.4+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'irony' if x == 1 else 'no irony')

labels = test['label'].unique()

test

,text,label
0,@user Can U Help?||More conservatives needed o...,no irony
1,"Just walked in to #Starbucks and asked for a ""...",irony
2,#NOT GONNA WIN,no irony
3,@user He is exactly that sort of person. Weirdo!,no irony
4,So much #sarcasm at work mate 10/10 #boring 10...,irony
...,...,...
779,"If you drag yesterday into today, your tomorro...",no irony
780,Congrats to my fav @user & her team & my birth...,no irony
781,@user Jessica sheds tears at her fan signing e...,no irony
782,#Irony: al jazeera is pro Anti - #GamerGate be...,irony


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_16152\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


95404032

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "deepseek-r1:1.5b",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 500
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()
    if '</think>' not in content:
        return 'error', response_time, vram_usage, ram_usage_bytes, total_time

    content = content.split('</think>')[1]
    
    if 'no irony' in content:
        content = 'no irony'
    elif 'irony' in content:
        content = 'irony'
    else:
        content = 'error'

    print(f"Text: {text}")
    print(f"Response: {content}")

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [7]:
test = test.groupby('label').head(17)

In [8]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_16152\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to
Response: no irony
Text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony
Response: irony
Text: #NOT GONNA WIN
Response: no irony
Text: @user He is exactly that sort of person. Weirdo!
Response: no irony
Text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm
Response: irony
Text: Corny jokes are my absolute favorite
Response: no irony
Text: People complain about my backround pic and all I feel is like "hey don't blame me, Albert E might have spoken those words" #sarcasm #life
Response: irony
Text: @user @user Darn, my sock joke needs fixing?
Response: no irony
Text: if Christian expects Fifa to sleep in my bed with me tonight, he's wrong 👿
Response: irony
Text: People who tell people with anxiety to "just stop worrying about it" are my favorite kind of people #not #educate

C:\Users\Rafael\AppData\Local\Temp\ipykernel_16152\2610005304.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)
C:\Users\Rafael\AppData\Local\Temp\ipykernel_16152\2610005304.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply

In [9]:
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
0,@user Can U Help?||More conservatives needed o...,no irony,no irony,4.973552,2433,90.988281,2.931220
1,"Just walked in to #Starbucks and asked for a ""...",irony,irony,4.464278,2402,90.488281,2.434293
2,#NOT GONNA WIN,no irony,no irony,4.322705,2401,91.011719,2.281778
3,@user He is exactly that sort of person. Weirdo!,no irony,no irony,4.297794,2402,91.015625,2.258633
4,So much #sarcasm at work mate 10/10 #boring 10...,irony,irony,4.908305,2440,90.468750,2.860175
5,Corny jokes are my absolute favorite,no irony,no irony,5.628904,2394,89.601562,3.579482
6,People complain about my backround pic and all...,irony,irony,4.425078,2397,90.179688,2.397651
7,"@user @user Darn, my sock joke needs fixing?",no irony,no irony,4.389089,2398,90.246094,2.339298
8,if Christian expects Fifa to sleep in my bed w...,no irony,irony,4.735307,2405,89.324219,2.690969
9,"People who tell people with anxiety to ""just s...",irony,no irony,4.623075,2390,89.832031,2.575122


In [10]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.647059
F1 score: 0.655000
Precision: 0.763587
Recall: 0.647059


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 4.699535958907184
Average VRAM usage: 2392.9411764705883
Average RAM usage: 90.03538602941177
Average total time: 2.6561635235294117


In [12]:
# save results to txt
with open('results/deepseekR1_ZS_binary3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')
    